In [ ]:
# Woolworths Proximity Tracker 🛒

**Goal:** Estimate how rural/urban a South African house is based on distance to the nearest Woolworths store.

**Core Features:**
- Geocode user-entered South African addresses
- Calculate distances to all Woolworths stores
- Differentiate between store formats (full-line, food stores, convenience, Engen Foodstops)
- Interactive mapping with Folium
- Urbanization proxy analysis
- Foundation for future house price modeling

**Data Source:** Publicly available Woolworths store locations

## 1. Setup & Imports

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import requests
from typing import Dict, List, Tuple, Optional
import json

# Geospatial libraries
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Mapping
import folium
from folium import plugins

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"📅 Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully!
📅 Notebook initialized: 2025-12-15 05:26:40


## 2. Woolworths Store Data Collection

We'll create a dataset of Woolworths stores across South Africa. In production, this would be scraped from the Woolworths store locator or obtained via API. For this implementation, we'll create a representative sample dataset with real store locations.

In [2]:
# Woolworths Store Format Types
STORE_TYPES = {
    'full_line': 'Woolworths Full-Line/Department Store',
    'food': 'Woolworths Food',
    'foodstop': 'Engen Woolworths Foodstop',
    'convenience': 'Woolworths Convenience',
    'market': 'Woolworths Market'
}

# Sample Woolworths store data across South Africa
# This is a representative dataset - in production, scrape from Woolworths store locator
woolworths_stores = [
    # Gauteng - Urban centers
    {'name': 'Sandton City', 'city': 'Sandton', 'province': 'Gauteng', 'lat': -26.1076, 'lon': 28.0567, 'type': 'full_line'},
    {'name': 'Rosebank Mall', 'city': 'Rosebank', 'province': 'Gauteng', 'lat': -26.1461, 'lon': 28.0420, 'type': 'food'},
    {'name': 'Menlyn Park', 'city': 'Pretoria', 'province': 'Gauteng', 'lat': -25.7842, 'lon': 28.2772, 'type': 'full_line'},
    {'name': 'Mall of Africa', 'city': 'Midrand', 'province': 'Gauteng', 'lat': -25.9341, 'lon': 28.1168, 'type': 'food'},
    {'name': 'Fourways Mall', 'city': 'Fourways', 'province': 'Gauteng', 'lat': -26.0158, 'lon': 28.0090, 'type': 'food'},
    {'name': 'Centurion Mall', 'city': 'Centurion', 'province': 'Gauteng', 'lat': -25.8607, 'lon': 28.1886, 'type': 'food'},
    {'name': 'Cresta Shopping Centre', 'city': 'Johannesburg', 'province': 'Gauteng', 'lat': -26.1289, 'lon': 27.9835, 'type': 'food'},
    {'name': 'Woodlands Boulevard', 'city': 'Pretoria', 'province': 'Gauteng', 'lat': -25.7518, 'lon': 28.3373, 'type': 'convenience'},
    
    # Western Cape - Cape Town region
    {'name': 'V&A Waterfront', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.9049, 'lon': 18.4189, 'type': 'full_line'},
    {'name': 'Canal Walk', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.8926, 'lon': 18.5091, 'type': 'full_line'},
    {'name': 'Cavendish Square', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.9728, 'lon': 18.4625, 'type': 'food'},
    {'name': 'Blue Route Mall', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -34.0285, 'lon': 18.4621, 'type': 'food'},
    {'name': 'Stellenbosch', 'city': 'Stellenbosch', 'province': 'Western Cape', 'lat': -33.9321, 'lon': 18.8602, 'type': 'food'},
    {'name': 'Somerset Mall', 'city': 'Somerset West', 'province': 'Western Cape', 'lat': -34.0789, 'lon': 18.8420, 'type': 'food'},
    {'name': 'Gardens Centre', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.9435, 'lon': 18.4087, 'type': 'market'},
    
    # KwaZulu-Natal - Durban region
    {'name': 'Gateway Theatre', 'city': 'Durban', 'province': 'KwaZulu-Natal', 'lat': -29.7706, 'lon': 31.0485, 'type': 'full_line'},
    {'name': 'The Pavilion', 'city': 'Durban', 'province': 'KwaZulu-Natal', 'lat': -29.8243, 'lon': 30.9381, 'type': 'food'},
    {'name': 'La Lucia Mall', 'city': 'Durban', 'province': 'KwaZulu-Natal', 'lat': -29.7779, 'lon': 31.0540, 'type': 'food'},
    {'name': 'Pietermaritzburg', 'city': 'Pietermaritzburg', 'province': 'KwaZulu-Natal', 'lat': -29.6001, 'lon': 30.3794, 'type': 'food'},
    
    # Eastern Cape
    {'name': 'Walmer Park', 'city': 'Port Elizabeth', 'province': 'Eastern Cape', 'lat': -33.9763, 'lon': 25.5821, 'type': 'food'},
    {'name': 'Hemingways Mall', 'city': 'East London', 'province': 'Eastern Cape', 'lat': -33.0154, 'lon': 27.9122, 'type': 'food'},
    {'name': 'Baywest Mall', 'city': 'Port Elizabeth', 'province': 'Eastern Cape', 'lat': -33.9237, 'lon': 25.5360, 'type': 'food'},
    
    # Free State
    {'name': 'Mimosa Mall', 'city': 'Bloemfontein', 'province': 'Free State', 'lat': -29.0974, 'lon': 26.1571, 'type': 'food'},
    {'name': 'Loch Logan Waterfront', 'city': 'Bloemfontein', 'province': 'Free State', 'lat': -29.1107, 'lon': 26.2185, 'type': 'food'},
    
    # Mpumalanga
    {'name': 'Riverside Mall', 'city': 'Nelspruit', 'province': 'Mpumalanga', 'lat': -25.4748, 'lon': 30.9783, 'type': 'food'},
    
    # North West
    {'name': 'Rustenburg', 'city': 'Rustenburg', 'province': 'North West', 'lat': -25.6672, 'lon': 27.2421, 'type': 'food'},
    
    # Limpopo
    {'name': 'Mall of the North', 'city': 'Polokwane', 'province': 'Limpopo', 'lat': -23.8655, 'lon': 29.4524, 'type': 'food'},
    
    # Convenience stores & Foodstops (examples)
    {'name': 'Engen Foodstop N1 North', 'city': 'Johannesburg', 'province': 'Gauteng', 'lat': -26.0000, 'lon': 28.1000, 'type': 'foodstop'},
    {'name': 'Engen Foodstop N2 Cape Town', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.9500, 'lon': 18.4800, 'type': 'foodstop'},
    {'name': 'Kloof Street', 'city': 'Cape Town', 'province': 'Western Cape', 'lat': -33.9319, 'lon': 18.4104, 'type': 'convenience'},
]

# Create DataFrame
df_stores = pd.DataFrame(woolworths_stores)

# Add store type descriptions
df_stores['type_description'] = df_stores['type'].map(STORE_TYPES)

print(f"📊 Loaded {len(df_stores)} Woolworths stores across South Africa")
print(f"\n🏪 Store type distribution:")
print(df_stores['type_description'].value_counts())
print(f"\n📍 Provincial distribution:")
print(df_stores['province'].value_counts())

df_stores.head(10)

📊 Loaded 30 Woolworths stores across South Africa

🏪 Store type distribution:
type_description
Woolworths Food                          20
Woolworths Full-Line/Department Store     5
Woolworths Convenience                    2
Engen Woolworths Foodstop                 2
Woolworths Market                         1
Name: count, dtype: int64

📍 Provincial distribution:
province
Gauteng          9
Western Cape     9
KwaZulu-Natal    4
Eastern Cape     3
Free State       2
Mpumalanga       1
North West       1
Limpopo          1
Name: count, dtype: int64


,name,city,province,lat,lon,type,type_description
0,Sandton City,Sandton,Gauteng,-26.1076,28.0567,full_line,Woolworths Full-Line/Department Store
1,Rosebank Mall,Rosebank,Gauteng,-26.1461,28.0420,food,Woolworths Food
2,Menlyn Park,Pretoria,Gauteng,-25.7842,28.2772,full_line,Woolworths Full-Line/Department Store
3,Mall of Africa,Midrand,Gauteng,-25.9341,28.1168,food,Woolworths Food
4,Fourways Mall,Fourways,Gauteng,-26.0158,28.0090,food,Woolworths Food
5,Centurion Mall,Centurion,Gauteng,-25.8607,28.1886,food,Woolworths Food
6,Cresta Shopping Centre,Johannesburg,Gauteng,-26.1289,27.9835,food,Woolworths Food
7,Woodlands Boulevard,Pretoria,Gauteng,-25.7518,28.3373,convenience,Woolworths Convenience
8,V&A Waterfront,Cape Town,Western Cape,-33.9049,18.4189,full_line,Woolworths Full-Line/Department Store
9,Canal Walk,Cape Town,Western Cape,-33.8926,18.5091,full_line,Woolworths Full-Line/Department Store


## 3. Geocoding Helper Functions

Functions to geocode South African addresses and handle various input formats.

In [3]:
import ssl
import certifi
from geopy.geocoders import Nominatim

class AddressGeocoder:
    """Geocode South African addresses with error handling and retries."""
    
    def __init__(self, user_agent: str = "woolworths_proximity_tracker"):
        # Create SSL context with certifi certificates
        ctx = ssl.create_default_context(cafile=certifi.where())
        self.geolocator = Nominatim(user_agent=user_agent, ssl_context=ctx)
        
    def geocode_address(self, address: str, country: str = "South Africa") -> Optional[Dict]:
        """
        Geocode a South African address.
        
        Args:
            address: Street address, suburb, or city
            country: Country name (default: South Africa)
            
        Returns:
            Dictionary with lat, lon, and full address, or None if geocoding fails
        """
        try:
            # Ensure South Africa is in the query
            if "south africa" not in address.lower():
                full_address = f"{address}, {country}"
            else:
                full_address = address
            
            # Geocode with timeout
            location = self.geolocator.geocode(full_address, timeout=10)
            
            if location:
                return {
                    'lat': location.latitude,
                    'lon': location.longitude,
                    'address': location.address,
                    'raw': location.raw
                }
            else:
                print(f"⚠️  Could not geocode: {full_address}")
                return None
                
        except GeocoderTimedOut:
            print(f"⏱️  Geocoding timed out for: {address}")
            return None
        except GeocoderServiceError as e:
            print(f"❌ Geocoding service error: {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            return None
    
    def geocode_coordinates(self, lat: float, lon: float) -> Optional[Dict]:
        """
        Reverse geocode coordinates to get address.
        
        Args:
            lat: Latitude
            lon: Longitude
            
        Returns:
            Dictionary with address information
        """
        try:
            location = self.geolocator.reverse((lat, lon), timeout=10)
            if location:
                return {
                    'address': location.address,
                    'raw': location.raw
                }
            return None
        except Exception as e:
            print(f"❌ Reverse geocoding error: {e}")
            return None

# Initialize geocoder
geocoder = AddressGeocoder()

print("✅ Geocoding functions initialized!")
print("📍 Ready to geocode South African addresses")

✅ Geocoding functions initialized!
📍 Ready to geocode South African addresses


## 4. Distance Calculation & Proximity Analysis

Calculate distances from a location to all Woolworths stores and analyze proximity patterns.

In [4]:
class WooliesProximityAnalyzer:
    """Analyze proximity to Woolworths stores and estimate urbanization level."""
    
    def __init__(self, stores_df: pd.DataFrame):
        self.stores_df = stores_df
        
    def calculate_distances(self, lat: float, lon: float) -> pd.DataFrame:
        """
        Calculate distances from a location to all stores.
        
        Args:
            lat: Latitude of target location
            lon: Longitude of target location
            
        Returns:
            DataFrame with stores and distances, sorted by proximity
        """
        target_location = (lat, lon)
        
        # Calculate distances
        distances = []
        for _, store in self.stores_df.iterrows():
            store_location = (store['lat'], store['lon'])
            distance_km = geodesic(target_location, store_location).kilometers
            distances.append(distance_km)
        
        # Add distances to dataframe
        result_df = self.stores_df.copy()
        result_df['distance_km'] = distances
        result_df['distance_m'] = result_df['distance_km'] * 1000
        
        # Sort by distance
        result_df = result_df.sort_values('distance_km').reset_index(drop=True)
        
        return result_df
    
    def get_nearest_stores(self, lat: float, lon: float, n: int = 5) -> pd.DataFrame:
        """Get the N nearest Woolworths stores."""
        all_distances = self.calculate_distances(lat, lon)
        return all_distances.head(n)
    
    def get_nearest_by_type(self, lat: float, lon: float) -> pd.DataFrame:
        """Get the nearest store of each type."""
        all_distances = self.calculate_distances(lat, lon)
        nearest_by_type = all_distances.groupby('type').first().reset_index()
        return nearest_by_type.sort_values('distance_km')
    
    def estimate_urbanization(self, lat: float, lon: float) -> Dict:
        """
        Estimate urbanization level based on Woolworths proximity.
        
        Uses distance to nearest store and store density as proxy for urbanization.
        
        Returns:
            Dictionary with urbanization metrics and classification
        """
        all_distances = self.calculate_distances(lat, lon)
        
        nearest_distance = all_distances['distance_km'].iloc[0]
        stores_within_5km = len(all_distances[all_distances['distance_km'] <= 5])
        stores_within_10km = len(all_distances[all_distances['distance_km'] <= 10])
        stores_within_25km = len(all_distances[all_distances['distance_km'] <= 25])
        
        # Classification logic
        if nearest_distance < 2 and stores_within_5km >= 3:
            urbanization_level = "Highly Urban"
            description = "Dense urban area with multiple nearby Woolworths stores"
        elif nearest_distance < 5 and stores_within_10km >= 2:
            urbanization_level = "Urban"
            description = "Urban area with good access to Woolworths"
        elif nearest_distance < 10:
            urbanization_level = "Suburban"
            description = "Suburban area with moderate access to Woolworths"
        elif nearest_distance < 25:
            urbanization_level = "Peri-urban"
            description = "Peri-urban or small town area"
        else:
            urbanization_level = "Rural"
            description = "Rural area with limited access to Woolworths"
        
        return {
            'urbanization_level': urbanization_level,
            'description': description,
            'nearest_distance_km': round(nearest_distance, 2),
            'nearest_store': all_distances.iloc[0]['name'],
            'nearest_store_type': all_distances.iloc[0]['type_description'],
            'stores_within_5km': stores_within_5km,
            'stores_within_10km': stores_within_10km,
            'stores_within_25km': stores_within_25km,
            'urbanization_score': self._calculate_urbanization_score(
                nearest_distance, stores_within_5km, stores_within_10km
            )
        }
    
    def _calculate_urbanization_score(self, nearest_distance: float, 
                                     stores_5km: int, stores_10km: int) -> float:
        """
        Calculate a continuous urbanization score (0-100).
        
        Higher score = more urban
        Based on: distance to nearest store + store density
        """
        # Distance component (0-50 points, inverse relationship)
        # 0km = 50 points, 50km+ = 0 points
        distance_score = max(0, 50 - nearest_distance)
        
        # Density component (0-50 points)
        # Based on stores within 5km and 10km
        density_score = min(50, (stores_5km * 10) + (stores_10km * 2))
        
        return round(distance_score + density_score, 1)

# Initialize analyzer
analyzer = WooliesProximityAnalyzer(df_stores)

print("✅ Proximity analyzer initialized!")
print("📏 Ready to calculate distances and estimate urbanization levels")

✅ Proximity analyzer initialized!
📏 Ready to calculate distances and estimate urbanization levels


## 5. Interactive Mapping with Folium

Create interactive maps showing the user's location, nearby Woolworths stores, and proximity circles.

In [5]:
class WooliesMapper:
    """Create interactive maps showing Woolworths proximity."""
    
    # Store type colors and icons
    STORE_COLORS = {
        'full_line': 'red',
        'food': 'blue',
        'foodstop': 'green',
        'convenience': 'orange',
        'market': 'purple'
    }
    
    STORE_ICONS = {
        'full_line': 'shopping-cart',
        'food': 'cutlery',
        'foodstop': 'car',
        'convenience': 'shopping-basket',
        'market': 'leaf'
    }
    
    def __init__(self, stores_df: pd.DataFrame):
        self.stores_df = stores_df
    
    def create_proximity_map(self, user_lat: float, user_lon: float, 
                            distances_df: pd.DataFrame,
                            zoom_start: int = 11) -> folium.Map:
        """
        Create an interactive map showing user location and nearby stores.
        
        Args:
            user_lat: User's latitude
            user_lon: User's longitude
            distances_df: DataFrame with stores and calculated distances
            zoom_start: Initial zoom level
            
        Returns:
            Folium map object
        """
        # Create base map centered on user location
        m = folium.Map(
            location=[user_lat, user_lon],
            zoom_start=zoom_start,
            tiles='OpenStreetMap'
        )
        
        # Add user location marker (home icon)
        folium.Marker(
            [user_lat, user_lon],
            popup=folium.Popup("<b>Your Location</b>", max_width=200),
            tooltip="Your Location",
            icon=folium.Icon(color='darkred', icon='home', prefix='fa')
        ).add_to(m)
        
        # Add proximity circles
        folium.Circle(
            [user_lat, user_lon],
            radius=5000,  # 5km
            popup='5km radius',
            color='blue',
            fill=True,
            fillOpacity=0.1,
            weight=1
        ).add_to(m)
        
        folium.Circle(
            [user_lat, user_lon],
            radius=10000,  # 10km
            popup='10km radius',
            color='green',
            fill=True,
            fillOpacity=0.05,
            weight=1
        ).add_to(m)
        
        # Add store markers
        for idx, store in distances_df.head(20).iterrows():  # Show nearest 20 stores
            store_color = self.STORE_COLORS.get(store['type'], 'gray')
            store_icon = self.STORE_ICONS.get(store['type'], 'shopping-cart')
            
            # Create popup with store info
            popup_html = f"""
            <div style="font-family: Arial; width: 200px;">
                <h4 style="margin-bottom: 5px;">{store['name']}</h4>
                <p style="margin: 2px 0;"><b>Type:</b> {store['type_description']}</p>
                <p style="margin: 2px 0;"><b>City:</b> {store['city']}, {store['province']}</p>
                <p style="margin: 2px 0;"><b>Distance:</b> {store['distance_km']:.2f} km</p>
            </div>
            """
            
            folium.Marker(
                [store['lat'], store['lon']],
                popup=folium.Popup(popup_html, max_width=250),
                tooltip=f"{store['name']} ({store['distance_km']:.1f}km)",
                icon=folium.Icon(color=store_color, icon=store_icon, prefix='fa')
            ).add_to(m)
        
        # Add lines from user to nearest stores
        for idx, store in distances_df.head(3).iterrows():  # Lines to 3 nearest
            folium.PolyLine(
                locations=[[user_lat, user_lon], [store['lat'], store['lon']]],
                color='red',
                weight=2,
                opacity=0.6,
                popup=f"Distance: {store['distance_km']:.2f} km"
            ).add_to(m)
        
        # Add layer control
        folium.LayerControl().add_to(m)
        
        return m
    
    def create_all_stores_map(self) -> folium.Map:
        """Create a map showing all Woolworths stores in South Africa."""
        # Center on South Africa
        m = folium.Map(
            location=[-29.0, 25.0],
            zoom_start=6,
            tiles='OpenStreetMap'
        )
        
        # Add all stores
        for _, store in self.stores_df.iterrows():
            store_color = self.STORE_COLORS.get(store['type'], 'gray')
            store_icon = self.STORE_ICONS.get(store['type'], 'shopping-cart')
            
            popup_html = f"""
            <div style="font-family: Arial; width: 200px;">
                <h4 style="margin-bottom: 5px;">{store['name']}</h4>
                <p style="margin: 2px 0;"><b>Type:</b> {store['type_description']}</p>
                <p style="margin: 2px 0;"><b>Location:</b> {store['city']}, {store['province']}</p>
            </div>
            """
            
            folium.Marker(
                [store['lat'], store['lon']],
                popup=folium.Popup(popup_html, max_width=250),
                tooltip=store['name'],
                icon=folium.Icon(color=store_color, icon=store_icon, prefix='fa')
            ).add_to(m)
        
        return m

# Initialize mapper
mapper = WooliesMapper(df_stores)

print("✅ Mapping functions initialized!")
print("🗺️  Ready to create interactive maps")

✅ Mapping functions initialized!
🗺️  Ready to create interactive maps


## 6. Main Analysis Function

Combine geocoding, distance calculation, and mapping into a single analysis function.

In [6]:
def analyze_woolies_proximity(address: str, show_map: bool = True) -> Dict:
    """
    Complete proximity analysis for a South African address.
    
    Args:
        address: South African address to analyze
        show_map: Whether to display the interactive map
        
    Returns:
        Dictionary with analysis results
    """
    print(f"🔍 Analyzing Woolworths proximity for: {address}")
    print("=" * 70)
    
    # Step 1: Geocode the address
    print("\n📍 Step 1: Geocoding address...")
    location = geocoder.geocode_address(address)
    
    if not location:
        print("❌ Could not geocode address. Please check and try again.")
        return None
    
    print(f"✅ Found location: {location['address']}")
    print(f"   Coordinates: {location['lat']:.4f}, {location['lon']:.4f}")
    
    # Step 2: Calculate distances to all stores
    print("\n📏 Step 2: Calculating distances to Woolworths stores...")
    distances_df = analyzer.calculate_distances(location['lat'], location['lon'])
    
    # Step 3: Get nearest stores
    nearest_stores = analyzer.get_nearest_stores(location['lat'], location['lon'], n=5)
    print(f"\n🏪 Top 5 Nearest Woolworths Stores:")
    for idx, store in nearest_stores.iterrows():
        print(f"   {idx + 1}. {store['name']} ({store['type_description']})")
        print(f"      📍 {store['city']}, {store['province']} - {store['distance_km']:.2f} km away")
    
    # Step 4: Get nearest store by type
    nearest_by_type = analyzer.get_nearest_by_type(location['lat'], location['lon'])
    print(f"\n🏷️  Nearest Store by Type:")
    for idx, store in nearest_by_type.iterrows():
        print(f"   • {store['type_description']}: {store['name']} ({store['distance_km']:.2f} km)")
    
    # Step 5: Estimate urbanization
    print("\n🏙️  Step 3: Estimating urbanization level...")
    urban_metrics = analyzer.estimate_urbanization(location['lat'], location['lon'])
    
    print(f"\n{'=' * 70}")
    print(f"📊 URBANIZATION ANALYSIS RESULTS")
    print(f"{'=' * 70}")
    print(f"🎯 Classification: {urban_metrics['urbanization_level']}")
    print(f"📝 Description: {urban_metrics['description']}")
    print(f"🔢 Urbanization Score: {urban_metrics['urbanization_score']}/100")
    print(f"\n📏 Distance Metrics:")
    print(f"   • Nearest store: {urban_metrics['nearest_distance_km']} km")
    print(f"   • {urban_metrics['nearest_store']} ({urban_metrics['nearest_store_type']})")
    print(f"\n🏪 Store Density:")
    print(f"   • Within 5 km: {urban_metrics['stores_within_5km']} stores")
    print(f"   • Within 10 km: {urban_metrics['stores_within_10km']} stores")
    print(f"   • Within 25 km: {urban_metrics['stores_within_25km']} stores")
    print(f"{'=' * 70}")
    
    # Step 6: Create map
    if show_map:
        print("\n🗺️  Step 4: Creating interactive map...")
        proximity_map = mapper.create_proximity_map(
            location['lat'], 
            location['lon'], 
            distances_df
        )
        print("✅ Map created successfully!")
        
        # Return results with map
        return {
            'location': location,
            'urbanization': urban_metrics,
            'nearest_stores': nearest_stores,
            'nearest_by_type': nearest_by_type,
            'all_distances': distances_df,
            'map': proximity_map
        }
    else:
        # Return results without map
        return {
            'location': location,
            'urbanization': urban_metrics,
            'nearest_stores': nearest_stores,
            'nearest_by_type': nearest_by_type,
            'all_distances': distances_df
        }

print("✅ Main analysis function ready!")
print("💡 Use: analyze_woolies_proximity('your address here')")

✅ Main analysis function ready!
💡 Use: analyze_woolies_proximity('your address here')


## 7. Example Analysis - Test with Sample Addresses

Let's test the system with various South African addresses representing different urbanization levels.

In [7]:
# Example 1: Highly Urban - Sandton, Johannesburg
print("🏙️  EXAMPLE 1: Highly Urban Area")
print("=" * 70)
result1 = analyze_woolies_proximity("Sandton City, Johannesburg, South Africa")

# Display the map
if result1 and 'map' in result1:
    display(result1['map'])

🏙️  EXAMPLE 1: Highly Urban Area
🔍 Analyzing Woolworths proximity for: Sandton City, Johannesburg, South Africa

📍 Step 1: Geocoding address...
✅ Found location: Sandton City, Alice Lane, Johannesburg Ward 103, Sandton, City of Johannesburg Metropolitan Municipality, Gauteng, 2031, South Africa
   Coordinates: -26.1090, 28.0524

📏 Step 2: Calculating distances to Woolworths stores...

🏪 Top 5 Nearest Woolworths Stores:
   1. Sandton City (Woolworths Full-Line/Department Store)
      📍 Sandton, Gauteng - 0.46 km away
   2. Rosebank Mall (Woolworths Food)
      📍 Rosebank, Gauteng - 4.24 km away
   3. Cresta Shopping Centre (Woolworths Food)
      📍 Johannesburg, Gauteng - 7.23 km away
   4. Fourways Mall (Woolworths Food)
      📍 Fourways, Gauteng - 11.21 km away
   5. Engen Foodstop N1 North (Engen Woolworths Foodstop)
      📍 Johannesburg, Gauteng - 12.99 km away

🏷️  Nearest Store by Type:
   • Woolworths Full-Line/Department Store: Sandton City (0.46 km)
   • Woolworths Food: Roseba

In [ ]:
# Example 2: Suburban - Cape Town Northern Suburbs
print("\n\n🏡 EXAMPLE 2: Suburban Area")
print("=" * 70)
result2 = analyze_woolies_proximity("Durbanville, Cape Town, South Africa")

if result2 and 'map' in result2:
    display(result2['map'])

In [ ]:
# Example 3: Rural - Small town
print("\n\n🌾 EXAMPLE 3: Rural/Small Town Area")
print("=" * 70)
result3 = analyze_woolies_proximity("Oudtshoorn, Western Cape, South Africa")

if result3 and 'map' in result3:
    display(result3['map'])

## 8. Interactive User Input

Now you can enter your own South African address to analyze!

In [ ]:
# 🏠 Enter your own South African address here!
# Examples:
# - "123 Main Street, Cape Town"
# - "Pretoria, Gauteng"
# - "Umhlanga, Durban"

user_address = "Stellenbosch, Western Cape"  # <-- Change this to your address

# Run the analysis
user_result = analyze_woolies_proximity(user_address)

# Display the map
if user_result and 'map' in user_result:
    display(user_result['map'])

## 9. Visualizations & Store Analysis

Analyze the distribution of stores and urbanization patterns.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Woolworths Store Distribution Analysis', fontsize=16, fontweight='bold')

# 1. Store types distribution
store_counts = df_stores['type_description'].value_counts()
axes[0, 0].bar(range(len(store_counts)), store_counts.values, color='steelblue')
axes[0, 0].set_xticks(range(len(store_counts)))
axes[0, 0].set_xticklabels(store_counts.index, rotation=45, ha='right')
axes[0, 0].set_title('Store Type Distribution')
axes[0, 0].set_ylabel('Number of Stores')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Provincial distribution
province_counts = df_stores['province'].value_counts()
axes[0, 1].barh(range(len(province_counts)), province_counts.values, color='coral')
axes[0, 1].set_yticks(range(len(province_counts)))
axes[0, 1].set_yticklabels(province_counts.index)
axes[0, 1].set_title('Stores by Province')
axes[0, 1].set_xlabel('Number of Stores')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Store type by province
type_province = pd.crosstab(df_stores['province'], df_stores['type'])
type_province.plot(kind='bar', stacked=True, ax=axes[1, 0], colormap='Set3')
axes[1, 0].set_title('Store Types by Province')
axes[1, 0].set_xlabel('Province')
axes[1, 0].set_ylabel('Number of Stores')
axes[1, 0].legend(title='Store Type', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Geographic distribution (scatter)
colors = df_stores['type'].map(mapper.STORE_COLORS)
axes[1, 1].scatter(df_stores['lon'], df_stores['lat'], 
                   c=colors, s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1, 1].set_title('Geographic Distribution of Stores')
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
axes[1, 1].grid(alpha=0.3)

# Add legend for store types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=mapper.STORE_COLORS[t], label=STORE_TYPES[t]) 
                   for t in mapper.STORE_COLORS.keys()]
axes[1, 1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

print("📊 Store distribution visualizations created!")

In [ ]:
# Show overview map of all stores
print("🗺️  Creating overview map of all Woolworths stores in South Africa...")
overview_map = mapper.create_all_stores_map()
display(overview_map)

## 10. Future Work: Housing Price Model Foundation

Framework for modeling housing prices as a function of Woolworths proximity (to be implemented).

In [8]:
class HousingPriceFeatureExtractor:
    """
    Extract Woolworths proximity features for housing price modeling.
    
    This class prepares features that can be used in a regression model
    to predict housing prices based on proximity to Woolworths stores.
    """
    
    def __init__(self, analyzer: WooliesProximityAnalyzer):
        self.analyzer = analyzer
    
    def extract_features(self, lat: float, lon: float) -> Dict:
        """
        Extract all Woolworths proximity features for a location.
        
        Features include:
        - Distance to nearest store (overall and by type)
        - Store density (5km, 10km, 25km radii)
        - Urbanization score
        - Provincial indicators
        
        These features can be used as independent variables in a 
        housing price prediction model.
        """
        # Calculate all distances
        distances_df = self.analyzer.calculate_distances(lat, lon)
        
        # Get nearest by type
        nearest_by_type = self.analyzer.get_nearest_by_type(lat, lon)
        
        # Get urbanization metrics
        urban_metrics = self.analyzer.estimate_urbanization(lat, lon)
        
        # Compile features
        features = {
            # Distance features
            'distance_nearest_store': distances_df['distance_km'].iloc[0],
            'distance_nearest_fullline': nearest_by_type[nearest_by_type['type'] == 'full_line']['distance_km'].values[0] 
                if 'full_line' in nearest_by_type['type'].values else None,
            'distance_nearest_food': nearest_by_type[nearest_by_type['type'] == 'food']['distance_km'].values[0] 
                if 'food' in nearest_by_type['type'].values else None,
            'distance_nearest_convenience': nearest_by_type[nearest_by_type['type'] == 'convenience']['distance_km'].values[0] 
                if 'convenience' in nearest_by_type['type'].values else None,
            
            # Density features
            'stores_within_5km': urban_metrics['stores_within_5km'],
            'stores_within_10km': urban_metrics['stores_within_10km'],
            'stores_within_25km': urban_metrics['stores_within_25km'],
            
            # Urbanization features
            'urbanization_score': urban_metrics['urbanization_score'],
            'urbanization_level': urban_metrics['urbanization_level'],
            
            # Store type diversity (within 10km)
            'store_types_within_10km': len(distances_df[distances_df['distance_km'] <= 10]['type'].unique()),
            
            # Average distance to 3 nearest stores
            'avg_distance_3_nearest': distances_df.head(3)['distance_km'].mean(),
            
            # Average distance to 5 nearest stores
            'avg_distance_5_nearest': distances_df.head(5)['distance_km'].mean(),
        }
        
        return features
    
    def create_feature_dataframe(self, locations: List[Tuple[float, float]]) -> pd.DataFrame:
        """
        Create a feature DataFrame for multiple locations.
        
        Args:
            locations: List of (lat, lon) tuples
            
        Returns:
            DataFrame with one row per location and all features as columns
        """
        features_list = []
        
        for lat, lon in locations:
            features = self.extract_features(lat, lon)
            features['lat'] = lat
            features['lon'] = lon
            features_list.append(features)
        
        return pd.DataFrame(features_list)

# Initialize feature extractor
feature_extractor = HousingPriceFeatureExtractor(analyzer)

print("✅ Housing price feature extractor initialized!")
print("\n📋 Features available for modeling:")
print("   • Distance to nearest store (overall and by type)")
print("   • Store density (5km, 10km, 25km radii)")
print("   • Urbanization score (0-100)")
print("   • Store type diversity")
print("   • Average distances to nearest stores")
print("\n💡 Next steps for housing price modeling:")
print("   1. Collect housing price data with coordinates")
print("   2. Extract Woolworths features for each property")
print("   3. Train regression model (e.g., Linear, Random Forest, XGBoost)")
print("   4. Validate model performance")
print("   5. Analyze feature importance")

✅ Housing price feature extractor initialized!

📋 Features available for modeling:
   • Distance to nearest store (overall and by type)
   • Store density (5km, 10km, 25km radii)
   • Urbanization score (0-100)
   • Store type diversity
   • Average distances to nearest stores

💡 Next steps for housing price modeling:
   1. Collect housing price data with coordinates
   2. Extract Woolworths features for each property
   3. Train regression model (e.g., Linear, Random Forest, XGBoost)
   4. Validate model performance
   5. Analyze feature importance


In [ ]:
# Example: Extract features for a sample location
sample_location = (-26.1076, 28.0567)  # Sandton
sample_features = feature_extractor.extract_features(sample_location[0], sample_location[1])

print("🏠 Sample Feature Extraction for Sandton:")
print("=" * 60)
for feature, value in sample_features.items():
    print(f"   {feature}: {value}")
    
# Create a DataFrame for easy viewing
feature_df = pd.DataFrame([sample_features])
print("\n📊 Features as DataFrame:")
display(feature_df.T)

## 11. Summary & Usage Guide

### 🎯 Key Features

1. **Geocoding**: Convert any South African address to coordinates
2. **Distance Calculation**: Calculate distances to all Woolworths stores
3. **Store Differentiation**: Recognize different store formats (full-line, food, convenience, foodstops)
4. **Urbanization Estimation**: Classify areas as Highly Urban, Urban, Suburban, Peri-urban, or Rural
5. **Interactive Mapping**: Folium maps with store markers, proximity circles, and distance lines
6. **Feature Extraction**: Extract features for housing price modeling

### 📝 Quick Start Guide

```python
# Analyze any South African address
result = analyze_woolies_proximity("Your Address Here, South Africa")

# Display the map
if result and 'map' in result:
    display(result['map'])

# Access the data
urbanization = result['urbanization']
nearest_stores = result['nearest_stores']
```

### 🔮 Future Enhancements

- **Data Collection**: Scrape live Woolworths store data from their website
- **More Store Types**: Add more granular store classifications
- **Housing Price Model**: Train ML model to predict prices based on proximity
- **Time Series**: Analyze how store openings/closures affect area development
- **Route Planning**: Calculate actual driving distances (not just straight-line)
- **Competitor Analysis**: Add Pick n Pay, Checkers, Spar locations
- **Socioeconomic Data**: Combine with census data for richer analysis